## Libraries

In [0]:
%pip install openpyxl
%restart_python

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from matplotlib.dates import date2num, WeekdayLocator, DateFormatter, MO
import seaborn as sns

## Data

In [0]:
directory = '/Volumes/cpx_ml_silver_dev/marketing_mix_modelling/data_v1/'

sem = pd.read_excel(directory + 'raw_data/SEM EXH & LBE Mar 2023 to Jun 2025 weekly report_v2.xlsx', skiprows=2)
datorama = pd.read_excel(directory + 'raw_data/Datorama Mar 2023 to June 2025 daily report.xlsx', skiprows=4, skipfooter=1)
snapchat = pd.read_csv(directory + 'raw_data/Snapchat EXH & LBE Campaign & ad level Mar 2023 to Jun 2025.csv')
ooh = pd.read_csv(directory + 'raw_data/lbe_ooh.csv')
directbuy = pd.read_csv(directory + 'raw_data/lbe_directbuy.csv')
email = pd.read_csv(directory + 'clean_data/email.csv')
transactions = pd.read_csv(directory + 'raw_data/lbe_transactions.csv')

# Splitting Datorama dataset
meta = datorama[datorama['Publisher']=='Meta'].copy()
programmatic = datorama[datorama['Channel']!='Social'].copy()
tiktok = datorama[datorama['Publisher']=='TikTok'].copy()
snapchat = snapchat[snapchat['Amount Spent']!=0]

In [0]:
sem['Week'] = pd.to_datetime(sem['Week']).dt.to_period('W-THU').dt.start_time
meta['week'] = pd.to_datetime(meta['Day']).dt.to_period('W-THU').dt.start_time
programmatic['week'] = pd.to_datetime(programmatic['Day']).dt.to_period('W-THU').dt.start_time
tiktok['week'] = pd.to_datetime(tiktok['Day']).dt.to_period('W-THU').dt.start_time
snapchat['week'] = pd.to_datetime(snapchat['Start Time']).dt.to_period('W-THU').dt.start_time
ooh['week'] = pd.to_datetime(ooh['week']).dt.to_period('W-THU').dt.start_time
directbuy['week'] = pd.to_datetime(directbuy['week']).dt.to_period('W-THU').dt.start_time
transactions['week'] = pd.to_datetime(transactions['date']).dt.to_period('W-THU').dt.start_time

for df in [sem, meta, programmatic, tiktok, snapchat, ooh, email, transactions]:
    df.columns = df.columns.str.lower().str.replace(' ', '_')
    df['week'] = pd.to_datetime(df['week'])

In [0]:
sem = sem[(sem['account_name'].str.contains('LBE'))&(sem['week'].between('2023-03-31', '2025-06-30'))]
meta = meta[(meta['raw_campaign_name'].str.contains('LBE'))&(meta['week'].between('2023-03-31', '2025-06-30'))]
programmatic = programmatic[(programmatic['raw_campaign_name'].str.contains('LBE'))&(programmatic['week'].between('2023-03-31', '2025-06-30'))]
tiktok = tiktok[(tiktok['raw_campaign_name'].str.contains('LBE'))&(tiktok['week'].between('2023-03-31', '2025-06-30'))]
snapchat = snapchat[(snapchat['campaign_name'].str.contains('LBE'))&(snapchat['week'].between('2023-03-31', '2025-06-30'))]
transactions =  transactions[(transactions['week'].between('2023-03-31', '2025-06-30'))]

## EDA

In [0]:
palette = ['#065CB1', '#E5C224', 'grey', '#CA0707', '#1F1F1F', '#3B73CB', '#5C8DE8']

### Email

In [0]:
tmp = email.copy()
tmp = tmp[tmp['week'].between('2023-03-31', '2025-03-31')]
tmp['lob'] = np.where(tmp['journey_name'].str.contains('-RECR|-PDM|LBE|Royal|Granv'), 'LBE', tmp['lob'])
tmp = tmp[tmp['lob']=='LBE']

sends = tmp['sent'].sum()
impressions = tmp['unique_opens'].sum()
clicks = tmp['unique_clicks'].sum()

fig = plt.figure(figsize=(7,6), dpi=240)

plt.fill_betweenx(y=[1, 3.8], x1=[10,12], x2=[8,6], color=palette[2]);
plt.fill_betweenx(y=[4, 6.8], x1=[12,14], x2=[6,4], color=palette[1]);
plt.fill_betweenx(y=[7, 9.8], x1=[14,16], x2=[4,2], color=palette[0]);

plt.xticks([],[]);
plt.yticks([2,5,8], ['Clicks', 'Opens', 'Sends']);

for y, value in zip([2,5, 8], [clicks, impressions, sends]):
    plt.text(9, y+0.25, f'{value:,.0f}', fontsize=11, fontweight="bold", color="white", ha="center");

### Summary

In [0]:
tmp1 = sem[['week', 'campaign', 'cost', 'impr.', 'clicks']].copy()
tmp2 = meta[['week', 'raw_campaign_name', 'investment', 'impressions', 'clicks']].copy()
tmp3 = programmatic[['week', 'raw_campaign_name', 'investment', 'impressions', 'clicks']].copy()
tmp4 = tiktok[['week', 'raw_campaign_name', 'investment', 'impressions', 'clicks']].copy()
tmp5 = snapchat[['week', 'campaign_name', 'amount_spent', 'paid_impressions', 'clicks']].copy()
tmp6 = ooh
tmp7 = directbuy

cols = ['week', 'campaign_name', 'spend', 'impressions', 'clicks']
channels = ['SEM', 'Meta', 'Programmatic', 'TikTok', 'Snapchat']
for df in [tmp1, tmp2, tmp3, tmp4, tmp5]:
    df.columns = cols
    df['channel'] = channels.pop(0)

lbe_df = pd.concat([tmp1, tmp2, tmp3, tmp4, tmp5], ignore_index=True).fillna(0)

col = lbe_df['campaign_name']
lbe_df['location_type'] = np.select(
    [col.str.contains('Toronto & Mississauga|South & West|\+SQ'),
     col.str.contains('Toronto|South Ed|Calgary|Deerfoot|Winnipeg|Brentwood|Burnaby|Vancouver'), 
     col.str.contains('HPL|High'), col.str.contains('West Ed|Square|Miss|London|Mason|John|Avalon|Barrie'), 
     col.str.contains('LPL|Low'), col.str.contains('Royalmount|Granville|TRR_NL'), 
     col.str.contains('Brampton|Whitby|Dart'), col.str.contains('Fair|PDM_NL'),
     col.str.contains('TRR|Rec|OOH'), col.str.contains('PDM|Playdium')],
    ['Multiple', 'HPL', 'HPL', 'LPL', 'LPL', 
     'NL - TRR', 'PDM', 'NL - PDM', 
     'Canada - TRR', 'Canada - PDM'],
    default='Multiple'
)

lbe_df = pd.concat([lbe_df, tmp6, tmp7], ignore_index=True)

In [0]:
transactions['location_type'] = np.select(
    [transactions['location'].str.contains('Roundhouse|South Ed|Deerfoot|Tuxedo|Brentwood'),
     transactions['location'].str.contains('West Ed|Square One|Barrie|Avalon|London'),
     transactions['location'].str.contains('Granville|Royal'),
     transactions['location'].str.contains('Dartmouth|Whitby|Orion'),
     transactions['location'].str.contains('Fairview')],
    ['HPL', 'LPL', 'NL - TRR', 'PDM', 'NL - PDM']
)

#### Channels - TRR

In [0]:
tmp = lbe_df[lbe_df['location_type'].isin(['Canada - TRR', 'HPL', 'LPL', 'NL - TRR'])].copy()
tmp = tmp.groupby('channel').agg({'spend': 'sum', 'impressions': 'sum', 'clicks': 'sum'}).sort_values('spend', ascending=False).copy()
tmp = tmp / tmp.sum() * 100

indices = np.arange(len(tmp))
fig = plt.figure(figsize=(8, 6), dpi=240)
bars1 = plt.bar(indices - 0.15, tmp['spend'], width=0.3, color=palette[0], zorder=2, label='Spend (% of Total)')
bars2 = plt.bar(indices + 0.15, tmp['impressions'], width=0.3, color=palette[1], zorder=2, label='Impressions (% of Total)')

for bar in bars1 + bars2:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 0.5, f'{yval:.0f}%', ha='center', va='bottom', fontsize=8)

plt.xlabel('Channel')
plt.ylabel('% of Total')
plt.xticks(indices, tmp.index)
plt.ylim(0, 100)
plt.grid(axis='y', alpha=0.25)
plt.legend(loc='upper left')
fig.patch.set_facecolor('white')
plt.show()

In [0]:
tmp = lbe_df[lbe_df['location_type'].isin(['Canada - TRR', 'HPL', 'LPL', 'NL - TRR'])].copy()
tmp = tmp[tmp['channel']!='OOH']
tmp = tmp.groupby('channel').agg({'spend': 'sum', 'impressions': 'sum', 'clicks': 'sum'}).sort_values('spend', ascending=False).copy()
tmp = tmp / tmp.sum() * 100

indices = np.arange(len(tmp))
fig = plt.figure(figsize=(8, 6), dpi=240)
bars1 = plt.bar(indices - 0.15, tmp['spend'], width=0.3, color=palette[0], zorder=2, label='Spend (% of Total)')
bars2 = plt.bar(indices + 0.15, tmp['clicks'], width=0.3, color=palette[1], zorder=2, label='Clicks (% of Total)')

for bar in bars1 + bars2:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 0.5, f'{yval:.0f}%', ha='center', va='bottom', fontsize=8)

plt.xlabel('Channel')
plt.ylabel('% of Total')
plt.xticks(indices, tmp.index)
plt.ylim(0, 100)
plt.grid(axis='y', alpha=0.25)
plt.legend(loc='upper left')
fig.patch.set_facecolor('white')
plt.show()

#### Channels - PDM

In [0]:
tmp = lbe_df[lbe_df['location_type'].str.contains('PDM')].copy()
tmp = tmp.groupby('channel').agg({'spend': 'sum', 'impressions': 'sum', 'clicks': 'sum'}).sort_values('spend', ascending=False).copy()
tmp = tmp / tmp.sum() * 100

indices = np.arange(len(tmp))
fig = plt.figure(figsize=(8, 6), dpi=240)
bars1 = plt.bar(indices - 0.15, tmp['spend'], width=0.3, color=palette[0], zorder=2, label='Spend (% of Total)')
bars2 = plt.bar(indices + 0.15, tmp['impressions'], width=0.3, color=palette[1], zorder=2, label='Impressions (% of Total)')

for bar in bars1 + bars2:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 0.5, f'{yval:.0f}%', ha='center', va='bottom', fontsize=8)

plt.xlabel('Channel')
plt.ylabel('% of Total')
plt.xticks(indices, tmp.index)
plt.ylim(0, 100)
plt.grid(axis='y', alpha=0.25)
plt.legend(loc='upper right')
fig.patch.set_facecolor('white')
plt.show()

In [0]:
tmp = lbe_df[lbe_df['location_type'].str.contains('PDM')].copy()
tmp = tmp.groupby('channel').agg({'spend': 'sum', 'impressions': 'sum', 'clicks': 'sum'}).sort_values('spend', ascending=False).copy()
tmp = tmp / tmp.sum() * 100

indices = np.arange(len(tmp))
fig = plt.figure(figsize=(8, 6), dpi=240)
bars1 = plt.bar(indices - 0.15, tmp['spend'], width=0.3, color=palette[0], zorder=2, label='Spend (% of Total)')
bars2 = plt.bar(indices + 0.15, tmp['clicks'], width=0.3, color=palette[1], zorder=2, label='Clicks (% of Total)')

for bar in bars1 + bars2:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 0.5, f'{yval:.0f}%', ha='center', va='bottom', fontsize=8)

plt.xlabel('Channel')
plt.ylabel('% of Total')
plt.xticks(indices, tmp.index)
plt.ylim(0, 100)
plt.grid(axis='y', alpha=0.25)
plt.legend(loc='upper right')
fig.patch.set_facecolor('white')
plt.show()

In [0]:
tmp = lbe_df[lbe_df['channel']!='OOH'].copy()
tmp['channel'] = np.where(tmp['location_type'].str.contains('PDM'), tmp['channel'] + '\nPDM', tmp['channel'] + '\nTRR')
tmp = tmp.groupby('channel').agg({'spend': 'sum', 'impressions': 'sum', 'clicks': 'sum'}).sort_values('spend', ascending=False)
scaled_impressions = tmp['impressions'] / tmp['impressions'].max() * 200 + 10
tmp['ctr_pct'] = tmp['clicks'] / tmp['impressions'] * 100

plt.figure(figsize=(8, 6), dpi=240)
plt.scatter(tmp['spend'], tmp['ctr_pct'], s=scaled_impressions, color=palette[0], zorder=2)

for i, txt in enumerate(tmp.index):
    if txt in ['Meta\nTRR', 'Meta\nPDM']:
        plt.annotate(f"{txt} ({tmp['ctr_pct'][i]:.1f}%)", (tmp['spend'][i]-70000, tmp['ctr_pct'][i]), fontsize=7, ha='right', va='center')
    else:
        plt.annotate(f"{txt} ({tmp['ctr_pct'][i]:.1f}%)", (tmp['spend'][i]+45000, tmp['ctr_pct'][i]), fontsize=7, ha='left', va='center')

plt.xlabel('Spend ($)')
plt.ylabel('CTR (%)')
plt.xlim(-5*10**4, 2.5*10**6)
plt.ylim(0, 1)
plt.gca().xaxis.set_major_formatter(mtick.EngFormatter(unit=''))
plt.grid(alpha=0.25)
plt.show()

#### LOB

In [0]:
tmp = lbe_df[~lbe_df['location_type'].isin(['Multiple', 'Canada - TRR', 'Canada - PDM'])].copy()
tmp = tmp.groupby('location_type').agg({'spend': 'sum', 'impressions': 'sum', 'clicks': 'sum'}).sort_values('spend', ascending=False)
tmp = tmp / tmp.sum() * 100

indices = np.arange(len(tmp))
fig = plt.figure(figsize=(8, 6), dpi=240)
bars1 = plt.bar(indices - 0.15, tmp['spend'], width=0.3, color=palette[0], zorder=2, label='Spend (% of Total)')
bars2 = plt.bar(indices + 0.15, tmp['impressions'], width=0.3, color=palette[1], zorder=2, label='Impressions (% of Total)')

for bar in bars1 + bars2:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 0.5, f'{yval:.0f}%', ha='center', va='bottom', fontsize=8)

plt.xlabel('Location Type')
plt.ylabel('% of Total')
plt.xticks(indices, tmp.index)
plt.ylim(0, 50)
plt.grid(axis='y', alpha=0.25)
plt.legend(loc='upper right')
fig.patch.set_facecolor('white')
plt.show()

In [0]:
tmp = lbe_df[~lbe_df['location_type'].isin(['Multiple', 'Canada - TRR', 'Canada - PDM'])].copy()
tmp = tmp.groupby('location_type').agg({'spend': 'sum', 'impressions': 'sum', 'clicks': 'sum'}).sort_values('spend', ascending=False)
tmp = tmp / tmp.sum() * 100

indices = np.arange(len(tmp))
fig = plt.figure(figsize=(8, 6), dpi=240)
bars1 = plt.bar(indices - 0.15, tmp['spend'], width=0.3, color=palette[0], zorder=2, label='Spend (% of Total)')
bars2 = plt.bar(indices + 0.15, tmp['clicks'], width=0.3, color=palette[1], zorder=2, label='Clicks (% of Total)')

for bar in bars1 + bars2:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 0.5, f'{yval:.0f}%', ha='center', va='bottom', fontsize=8)

plt.xlabel('Location Type')
plt.ylabel('% of Total')
plt.xticks(indices, tmp.index)
plt.ylim(0, 50)
plt.grid(axis='y', alpha=0.25)
plt.legend(loc='upper right')
fig.patch.set_facecolor('white')
plt.show()

In [0]:
# Playdiium spend vs transactions
tmp1 = lbe_df.groupby(['week']).agg({'spend': 'sum', 'impressions': 'sum'}).copy()
tmp1['cpm'] = tmp1['spend'] / tmp1['impressions'] * 1000
tmp1['cpm'] = np.where(tmp1['cpm']>np.percentile(tmp1['cpm'], 95), np.percentile(tmp1['cpm'], 95), tmp1['cpm'])
tmp1 = tmp1.rename(columns={'spend': 'media_spend'})
tmp2 = transactions.groupby('week').agg({'transactions': 'sum', 'revenue': 'sum'}).copy()

tmp = pd.merge(tmp1, tmp2, how='right', left_index=True, right_index=True).fillna(0)

plt.figure(figsize=(8, 6), dpi=240)
plt.bar(tmp.index, tmp['media_spend'], color=palette[0], width=5, zorder=2, label='Media Spend ($)')
plt.plot(tmp.index, tmp['transactions'], color=palette[1], zorder=2, label='Transactions')
plt.gca().yaxis.set_major_formatter(mtick.EngFormatter())
plt.ylim(0, 2.5*10**5)
plt.xlabel('Week')
plt.ylabel('Amount')
plt.legend(loc='upper left')
plt.grid(axis='y', alpha=0.25)
plt.show()

print(tmp.corr()['media_spend']['transactions'])

In [0]:
tmp1 = transactions[transactions['week'].between('2023-12-29', '2024-06-28')].copy()
tmp2 = transactions[transactions['week'].between('2024-12-27', '2025-06-27')].copy()
tmp1 = tmp1.groupby(['location', 'location_type']).agg({'revenue': 'sum'}).rename(columns={'revenue': 'revenue1'})
tmp2 = tmp2.groupby(['location', 'location_type']).agg({'revenue': 'sum'}).rename(columns={'revenue': 'revenue2'})
tmp = pd.concat([tmp1, tmp2], axis=1).reset_index().dropna()
tmp['location'] = tmp['location'].str.replace('The Rec Room', 'TRR').str.replace('Playdium', 'PDM')
tmp['yoy_change'] = 100 * (tmp['revenue2'] / tmp['revenue1']) - 100
tmp = tmp.sort_values('yoy_change')

colors = [palette[1] if 'PDM' in loc else palette[0] for loc in tmp['location']]

plt.figure(figsize=(8, 6), dpi=240)
plt.barh(tmp['location'], tmp['yoy_change'], color=colors, zorder=2)
plt.gca().xaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
plt.xlim(-30, 10)
plt.xlabel('YoY Change in Revenue')
plt.grid(axis='x', alpha=0.25)
plt.legend(['Playdium Location'])
plt.tight_layout()
plt.show()

In [0]:
tmp1 = lbe_df.groupby(['week', 'location_type']).agg({'spend': 'sum'}).reset_index().copy()
tmp2 = transactions.groupby(['week', 'location_type']).agg({'transactions': 'sum'}).reset_index().copy()
tmp = pd.merge(tmp1, tmp2, how='right', on=['week', 'location_type'])
tmp = tmp[tmp['spend']>0].groupby('week').agg({'spend': 'sum', 'transactions': 'sum'})

# Best fit line
m, b = np.polyfit(tmp['spend'], tmp['transactions'], 1)
x_fit = np.linspace(1000, 4.9*10**4, 100)
y_fit = m * x_fit + b

plt.figure(figsize=(8, 6), dpi=240)
plt.scatter(tmp['spend'], tmp['transactions'], color=palette[0], s=3, zorder=2, label='Week')
plt.plot(x_fit, y_fit, color=palette[1], linestyle='--', zorder=3, label='Best Fit')
plt.xlim(0, 5*10**4)
plt.ylim(0, 2.5*10**5)
plt.xlabel('Spend')
plt.ylabel('Transactions')
plt.gca().xaxis.set_major_formatter(mtick.EngFormatter())
plt.gca().yaxis.set_major_formatter(mtick.EngFormatter())
plt.grid(alpha=0.25)
plt.legend()
plt.show()

In [0]:
# # Location type spend vs transactions
# tmp1 = lbe_df[lbe_df['location_type']!='Multiple'].groupby(['week', 'location_type']).agg({'spend': 'sum'}).reset_index().copy()
# tmp2 = transactions.groupby('week').agg({'transactions': 'sum'}).reset_index().copy()

# tmp = pd.merge(tmp1, tmp2, how='right', on='week').set_index('week')

# g = sns.FacetGrid(tmp, col='location_type', col_wrap=3, height=4)
# g.fig.set_size_inches(8, 6)
# g.fig.set_dpi(240)
# g.map(sns.scatterplot, 'spend', 'transactions', s=7, edgecolor=None, color=palette[0])
# g.set_axis_labels("Spend ($)", "Transactions")
# g.set_titles("{col_name}")
# for ax in g.axes.flatten(): 
#     ax.grid(True, alpha=0.25)
#     ax.xaxis.set_major_formatter(mtick.EngFormatter())
#     ax.yaxis.set_major_formatter(mtick.EngFormatter())
# plt.show()

#### Flighting

In [0]:
tmp = lbe_df[lbe_df['location_type']!='Multiple'].copy()
# tmp['location_type'] = np.where(tmp['campaign_name'].str.contains('Always On|AO'), tmp['location_type'] + ' (AO)', tmp['location_type'])
tmp = tmp.groupby(['week', 'location_type']).agg({'spend': 'sum'}).reset_index()

windows = {}

for loc, grp in tmp.groupby(['location_type']):
    grp = grp.sort_values('week')
    runs = []
    run_start = None
    prev_week = None

    for idx, row in grp.iterrows():
        week = row['week']
        if row['spend'] > 0:
            if run_start is None:
                run_start = week
            elif (week - prev_week).days > 8:
                duration = (prev_week + pd.Timedelta(days=7) - run_start).days
                runs.append((date2num(run_start), duration))
                run_start = week
            prev_week = week
        else:
            if run_start is not None and prev_week is not None:
                duration = (prev_week + pd.Timedelta(days=7) - run_start).days
                runs.append((date2num(run_start), duration))
                run_start = None
                prev_week = None
    # Finalize last run
    if run_start is not None and prev_week is not None:
        duration = (prev_week + pd.Timedelta(days=7) - run_start).days
        runs.append((date2num(run_start), duration))

    windows[loc] = runs

fig, ax = plt.subplots(figsize=(8, 6), dpi=240)

sorted_keys = sorted(windows.keys(), key=lambda x: (x[0], x[1]))

yticks = []
ylabels = []
height = 0.8
pad = 0.2

c = [plt.cm.tab10(0), plt.cm.tab10(0), plt.cm.tab10(1), plt.cm.tab10(2), plt.cm.tab10(3),
     plt.cm.tab10(3), plt.cm.tab10(4), plt.cm.tab10(4), plt.cm.tab10(8), plt.cm.tab10(10),
     plt.cm.tab10(10)]

for i, key in enumerate(sorted_keys):
    loc = key
    xranges = windows[key]
    y = i * (height + pad)
    color = c[i]
    ax.broken_barh(
        xranges,
        (y, height),
        facecolors=color,
        alpha=1,
        label=f"{loc}",
        zorder=2
    )
    yticks.append(y + height/2)
    ylabels.append(f"{loc}")

ax.set_ylim(-pad, yticks[-1] + height + pad)
ax.set_xlim(date2num(tmp['week'].min())-7, date2num(tmp['week'].max())+14)
ax.set_yticks(yticks)
ax.set_yticklabels(ylabels)
ax.invert_yaxis()

# show weeks on x‑axis
ax.xaxis_date()
# fig.autofmt_xdate()

# ax.set_title("Spend Flighting by Location Type")
ax.set_xlabel("Week")
ax.grid(axis='x', alpha=0.25)

plt.tight_layout()
plt.show()

In [0]:
tmp = lbe_df[lbe_df['location_type'].isin(['HPL', 'LPL', 'PDM'])][lbe_df['channel']!='OOH'].copy()
tmp['year'] = np.where(tmp['week'] <= '2024-06-30', 1, 2)
tmp = tmp.groupby(['location_type', 'year']).agg({'spend': 'sum', 'clicks': 'sum'})
tmp['cpa'] = tmp['spend'] / tmp['clicks']
tmp = tmp.unstack()['cpa']
tmp.columns = ['year1_cpa', 'year2_cpa']
tmp.index.name = 'location_type'

plt.figure(figsize=(8, 6), dpi=240)
bar1 = plt.bar(np.arange(len(tmp))-0.2, tmp['year1_cpa'], color=palette[0], width=0.4, label='Q2 2023 - Q2 2024', zorder=2)
bar2 = plt.bar(np.arange(len(tmp))+0.2, tmp['year2_cpa'], color=palette[1], width=0.4, label='Q3 2024 - Q2 2025', zorder=2)

for bar in list(bar1) + list(bar2):
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, yval+0.01, f'${yval:.2f}', ha='center', va='bottom', fontsize=8)

plt.xticks([0, 1, 2], ['HPL', 'LPL', 'PDM'])
plt.xlabel('Location Type')
plt.ylabel('CPA ($)')
plt.ylim(0, 1.4)
plt.legend(loc='upper left')
plt.grid(axis='y', alpha=0.25)
plt.show()

In [0]:
# Granville analysis
tmp1 = lbe_df[(lbe_df['location_type']=='NL - TRR')&(lbe_df['campaign_name'].str.contains('Gran'))].groupby('week').agg({'spend': 'sum', 'clicks': 'sum'})
tmp2 = transactions[['week', 'transactions']].groupby('week').sum().copy()
tmp = tmp1.merge(tmp2, how='right', left_index=True, right_index=True).fillna(0)
tmp = tmp[tmp.index>='2024-09-27']

fig, ax1 = plt.subplots(figsize=(9, 7), dpi=240)
ax2 = ax1.twinx()
ax1.bar(tmp.index, tmp.spend, color=palette[0], width=5, zorder=2, label='Spend ($)')
plt.plot(tmp.index, tmp.clicks, color=palette[1], zorder=2, label='Clicks')
ax1.set_ylabel('Spend ($)')
ax2.set_ylabel('Clicks')
ax1.yaxis.set_major_formatter(mtick.EngFormatter())
ax2.yaxis.set_major_formatter(mtick.EngFormatter())
ax1.set_ylim(0, 1.6*10**4)
ax2.set_ylim(0, 8*10**3)
ax1.set_yticks(np.arange(0, 17000, 2000))
ax2.set_yticks(np.arange(0, 9000, 1000))
ax1.legend(loc='upper left')
ax2.legend()
ax1.grid(axis='y', alpha=0.25)
ax1.set_xlabel('Week')
plt.show()

In [0]:
# Royalmount analysis
tmp1 = lbe_df[(lbe_df['location_type']=='NL - TRR')&(lbe_df['campaign_name'].str.contains('Royal'))].groupby('week').agg({'spend': 'sum', 'clicks': 'sum'})
tmp2 = transactions[['week', 'transactions']].groupby('week').sum().copy()
tmp = tmp1.merge(tmp2, how='right', left_index=True, right_index=True).fillna(0)
tmp = tmp[tmp.index>='2024-09-27']

fig, ax1 = plt.subplots(figsize=(9, 7), dpi=240)
ax2 = ax1.twinx()
ax1.bar(tmp.index, tmp.spend, color=palette[0], width=5, zorder=2, label='Spend ($)')
plt.plot(tmp.index, tmp.clicks, color=palette[1], zorder=2, label='Clicks')
ax1.set_ylabel('Spend ($)')
ax2.set_ylabel('Clicks')
ax1.yaxis.set_major_formatter(mtick.EngFormatter())
ax2.yaxis.set_major_formatter(mtick.EngFormatter())
ax1.set_ylim(0, 1.6*10**4)
ax2.set_ylim(0, 8*10**3)
ax1.set_yticks(np.arange(0, 17000, 2000))
ax2.set_yticks(np.arange(0, 9000, 1000))
ax1.legend(loc='upper left')
ax2.legend()
ax1.grid(axis='y', alpha=0.25)
ax1.set_xlabel('Week')
plt.show()

#### LOB - TRR

In [0]:
# Rec Room spend vs transactions
tmp1 = lbe_df[~lbe_df['location_type'].str.contains('PDM')].groupby(['week']).agg({'spend': 'sum', 'impressions': 'sum'}).copy()
tmp1 = tmp1.rename(columns={'spend': 'media_spend'})
tmp2 = transactions[transactions['lob']!='PDM'].groupby('week').agg({'transactions': 'sum', 'revenue': 'sum'}).copy()

tmp = pd.merge(tmp1, tmp2, how='right', left_index=True, right_index=True).fillna(0)

plt.figure(figsize=(8, 6), dpi=240)
plt.bar(tmp.index, tmp['media_spend'], color=palette[0], width=5, zorder=2, label='Media Spend ($)')
plt.plot(tmp.index, tmp['transactions'], color=palette[1], zorder=2, label='Transactions')
plt.gca().yaxis.set_major_formatter(mtick.EngFormatter())
plt.ylim(0, 2.5*10**5)
plt.xlabel('Week')
plt.ylabel('Amount')
plt.legend(loc='upper left')
plt.grid(axis='y', alpha=0.25)
plt.show()

print(tmp.corr()['media_spend']['transactions'])

In [0]:
# # Rec Room spend vs transactions
# tmp1 = lbe_df[~lbe_df['location_type'].str.contains('PDM')].groupby(['week']).agg({'spend': 'sum', 'impressions': 'sum'}).copy()
# tmp1['cpm'] = tmp1['spend'] / tmp1['impressions'] * 1000
# tmp1['cpm'] = np.where(tmp1['cpm']>np.percentile(tmp1['cpm'], 95), np.percentile(tmp1['cpm'], 95), tmp1['cpm'])
# tmp1 = tmp1.rename(columns={'spend': 'media_spend'})
# tmp2 = transactions[transactions['lob']!='PDM'].groupby('week').agg({'transactions': 'sum', 'revenue': 'sum'}).copy()

# tmp = pd.merge(tmp1, tmp2, how='right', left_index=True, right_index=True).fillna(0)

# fig, ax1 = plt.subplots(figsize=(8, 6), dpi=240)
# ax2 = ax1.twinx()
# ax1.bar(tmp.index, tmp['media_spend'], color=palette[0], width=5, zorder=2, label='Media Spend ($)')
# ax1.plot(tmp.index, tmp['transactions'], color=palette[1], zorder=2, label='Transactions')
# ax2.plot(tmp.index, tmp['cpm'], color=palette[4], zorder=2, label='CPM ($)')
# ax1.yaxis.set_major_formatter(mtick.EngFormatter())
# ax1.set_ylim(0, 5*10**5)
# ax2.set_ylim(0, 10)
# ax1.set_xlabel('Week')
# ax1.set_ylabel('Amount')
# ax2.set_ylabel('CPM ($)')
# ax1.legend(loc='upper left')
# ax2.legend(loc='upper right')
# plt.grid(axis='y', alpha=0.25)
# plt.show()

# print(tmp.corr()['media_spend']['transactions'])

#### LOB - PDM

In [0]:
# Playdiium spend vs transactions
tmp1 = lbe_df[lbe_df['location_type'].str.contains('PDM')].groupby(['week']).agg({'spend': 'sum', 'impressions': 'sum'}).copy()
tmp1 = tmp1.rename(columns={'spend': 'media_spend'})
tmp2 = transactions[transactions['lob']!='PDM'].groupby('week').agg({'transactions': 'sum', 'revenue': 'sum'}).copy()

tmp = pd.merge(tmp1, tmp2, how='right', left_index=True, right_index=True).fillna(0)

plt.figure(figsize=(8, 6), dpi=240)
plt.bar(tmp.index, tmp['media_spend'], color=palette[0], width=5, zorder=2, label='Media Spend ($)')
plt.plot(tmp.index, tmp['transactions'], color=palette[1], zorder=2, label='Transactions')
plt.gca().yaxis.set_major_formatter(mtick.EngFormatter())
plt.ylim(0, 2.5*10**5)
plt.xlabel('Week')
plt.ylabel('Amount')
plt.legend(loc='upper left')
plt.grid(axis='y', alpha=0.25)
plt.show()

print(tmp.corr()['media_spend']['transactions'])

In [0]:
# # Playdium spend vs transactions
# tmp1 = lbe_df[lbe_df['location_type'].str.contains('PDM')].groupby(['week']).agg({'spend': 'sum', 'impressions': 'sum'}).copy()
# tmp1['cpm'] = tmp1['spend'] / tmp1['impressions'] * 1000
# tmp1['cpm'] = np.where(tmp1['cpm']>np.percentile(tmp1['cpm'], 95), np.percentile(tmp1['cpm'], 95), tmp1['cpm'])
# tmp1 = tmp1.rename(columns={'spend': 'media_spend'})
# tmp2 = transactions[transactions['lob']=='PDM'].groupby('week').agg({'transactions': 'sum', 'revenue': 'sum'}).copy()

# tmp = pd.merge(tmp1, tmp2, how='right', left_index=True, right_index=True).fillna(0)

# fig, ax1 = plt.subplots(figsize=(8, 6), dpi=240)
# ax2 = ax1.twinx()
# ax1.bar(tmp.index, tmp['media_spend'], color=palette[0], width=5, zorder=2, label='Media Spend ($)')
# ax1.plot(tmp.index, tmp['transactions'], color=palette[1], zorder=2, label='Transactions')
# ax2.plot(tmp.index, tmp['cpm'], color=palette[4], zorder=2, label='CPM ($)')
# ax1.yaxis.set_major_formatter(mtick.EngFormatter())
# ax1.set_ylim(0, 7*10**4)
# ax2.set_ylim(0, 14)
# ax1.set_xlabel('Week')
# ax1.set_ylabel('Amount')
# ax2.set_ylabel('CPM ($)')
# ax1.legend(loc='upper left')
# ax2.legend(loc='upper right')
# plt.grid(axis='y', alpha=0.25)
# plt.show()

# print(tmp.corr()['media_spend']['transactions'])